# 第8章 智能机器人系统开发

本 notebook 对应《第8章-智能机器人系统开发》的 **8.1 系统概述**、**8.2 本体开发** 与 **8.3 SLAM开发** 三节内容，带你从'三大核心模块'出发，理解差动驱动机器人的运动控制闭环、ROS2 话题机制、多传感器融合、激光 SLAM 与 Nav2 导航的完整技术栈。

**运行环境**：`cann_9.0.0-py3.11-A2-arm-20260715` · `ASCEND, 1*NPU 910B3, 16vCPUs, 32GiB`

---

## 目录

1. 智能机器人系统概述
2. 智能机器人本体开发
3. 智能机器人SLAM开发
4. 小结
5. 课后练习

---

## 1. 智能机器人系统概述

智能机器人系统是一个高度复杂的综合体系，其核心由紧密协作的**三大模块**构成。硬件系统提供物理基础，软件系统实现系统管理与任务调度，智能算法赋予机器人真正的'智能'。三者深度融合，共同实现了机器人从简单的自动化执行到复杂的智能化决策的跨越。

<img src="../../images/robot_system_architecture.png" alt="机器人系统架构" style="display: block; margin-left: 0;" />

### 1.1 三大核心模块

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">模块</th>
<th style="text-align: left;">角色</th>
<th style="text-align: left;">关键组成</th>
</tr>
<tr>
<td style="text-align: left;">硬件系统</td>
<td style="text-align: left;">机器人的'身体'</td>
<td style="text-align: left;">机械结构、传感器（摄像头/激光雷达/IMU）、驱动执行、控制决策</td>
</tr>
<tr>
<td style="text-align: left;">软件系统</td>
<td style="text-align: left;">机器人的'神经系统'</td>
<td style="text-align: left;">操作系统层、中间件（ROS2/DDS）、功能模块</td>
</tr>
<tr>
<td style="text-align: left;">智能算法</td>
<td style="text-align: left;">机器人的'大脑'</td>
<td style="text-align: left;">机器学习、计算机视觉、路径规划、SLAM、深度强化学习</td>
</tr>
</table>

### 1.2 硬件系统详解

以典型的**差动驱动移动机器人**为例，其硬件体系自上而下分为四个核心层级：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">层级</th>
<th style="text-align: left;">模块</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;">01</td>
<td style="text-align: left;">机械本体结构</td>
<td style="text-align: left;">板式框架，两个独立驱动轮 + 万向脚轮</td>
</tr>
<tr>
<td style="text-align: left;">02</td>
<td style="text-align: left;">驱动执行模块</td>
<td style="text-align: left;">直流减速电机 + 行星减速器 + H桥驱动器</td>
</tr>
<tr>
<td style="text-align: left;">03</td>
<td style="text-align: left;">运动控制模块</td>
<td style="text-align: left;">STM32 微控制器，编码器读取 + PID 闭环</td>
</tr>
<tr>
<td style="text-align: left;">04</td>
<td style="text-align: left;">智能决策模块</td>
<td style="text-align: left;">昇腾开发板/树莓派/Jetson，运行 Linux + ROS2</td>
</tr>
</table>

<img src="../../images/differential_drive_kinematics.png" alt="差动驱动运动学" style="display: block; margin-left: 0;" />

### 1.3 ROS2 机器人操作系统

ROS2 相比 ROS1 的革命性改进在于采用 **DDS（数据分发服务）** 作为底层通信中间件：

- **去中心化**：无需 roscore 主节点，节点间自动发现
- **丰富的 QoS 策略**：精细控制数据传输的可靠性与实时性
- **内置安全特性**：支持加密、身份认证
- **跨平台**：支持 Linux / Windows / macOS / RTOS

ROS2 的四层架构：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">层级</th>
<th style="text-align: left;">名称</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;">L1</td>
<td style="text-align: left;">操作系统层</td>
<td style="text-align: left;">跨平台支持，含 RTOS</td>
</tr>
<tr>
<td style="text-align: left;">L2</td>
<td style="text-align: left;">中间件层</td>
<td style="text-align: left;">DDS 实现通信</td>
</tr>
<tr>
<td style="text-align: left;">L3</td>
<td style="text-align: left;">客户端库层</td>
<td style="text-align: left;">rclcpp / rclpy</td>
</tr>
<tr>
<td style="text-align: left;">L4</td>
<td style="text-align: left;">应用层</td>
<td style="text-align: left;">功能节点与功能包</td>
</tr>
</table>

### 1.4 基于昇腾硬件的智能巡检机器人

华为昇腾硬件开发板以内置的**达芬奇架构 NPU**，结合 ROS2 软件框架，可构建'感知更智能、决策更精准'的巡检机器人系统：

- **感知层**：激光雷达 -> `/scan`，IMU -> `/imu/data`
- **AI 处理层**：昇腾 NPU 运行 YOLO/SSD/DeepLab，30fps+ 实时检测 -> **语义 SLAM**
- **规划层**：全局路径规划订阅语义地图，选择'智能'路径
- **协作层**：多台机器人通过 DDS 自动发现，共享语义地图片段

> **昇腾 NPU 的价值**：为深度学习模型提供专用算力，使机器人能实时进行视觉识别、语义分割等高级认知任务——这是具身智能中'智能'的核心。

### 1.5 具身智能

**机器人智能系统**是实现具身智能的具体技术架构与工程系统，**具身智能**是指导机器人智能系统设计的核心思想与终极范式。

- **ROS2**：具身交互的'分布式神经系统'——节点/话题/服务为'身体'与'智能'建立高带宽、低延迟的神经通路
- **昇腾 NPU**：具身认知的'大脑'——为多模态感知数据的实时理解与决策提供算力
- **从'静态学习'到'交互学习'**：强化学习、模仿学习让机器人通过'试错'自主优化策略

---

## 2. 智能机器人本体开发

本节聚焦机器人运动控制的完整闭环、ROS2 话题机制与多传感器融合。

### 2.1 运动控制完整闭环

机器人的运动控制是一个'决策-规划-执行'贯穿多个层级的闭环过程：

<img src="../../images/motion_control_loop.png" alt="运动控制闭环" style="display: block; margin-left: 0;" />

**差动驱动运动学公式**（将整车线速度 $v$ 和角速度 $\omega$ 分解为左右轮速度）：

$$v_{right} = v + \frac{\omega \times L}{2}$$
$$v_{left} = v - \frac{\omega \times L}{2}$$

其中 $L$ 为轮距。

#### 动手实验：差动驱动运动学解算

下面用 Python 实现差动驱动运动学逆解算，并在 **昇腾 NPU** 上加速计算。

In [ ]:
import numpy as np

# 差动驱动运动学逆解算：整车速度 -> 左右轮速度
def differential_drive_inverse(v, omega, wheel_base):
    v_left = v - omega * wheel_base / 2
    v_right = v + omega * wheel_base / 2
    return v_left, v_right

# 场景：机器人以 0.3 m/s 前进，同时以 0.1 rad/s 左转
v = 0.3       # m/s
omega = 0.1   # rad/s
L = 0.28      # 轮距 0.28m

v_left, v_right = differential_drive_inverse(v, omega, L)
print(f'整车: v={v} m/s, omega={omega} rad/s, 轮距L={L} m')
print(f'左轮线速度: {v_left:.4f} m/s')
print(f'右轮线速度: {v_right:.4f} m/s')
print(f'验证: (v_r+v_l)/2 = {(v_right+v_left)/2:.4f} m/s (应等于 v)')

### 2.2 ROS2 话题机制

#### `/cmd_vel`：自上而下的运动指令通道

导航规划模块通过 `/cmd_vel` 话题发布 `geometry_msgs/msg/Twist` 消息：

- `Twist.linear.x`：前进/后退速度（m/s）
- `Twist.angular.z`：自转角速度（rad/s）

#### `/odom`：自下而上的状态反馈通道

底层 STM32 上传编码器 + IMU 数据，经 EKF 融合后发布 `nav_msgs/msg/Odometry` 消息：

- `pose`：位姿（x, y, 朝向四元数）
- `twist`：瞬时速度（线速度、角速度）

### 2.3 里程计数据计算

以 PPT 中的具体参数为例，从编码器脉冲到 ROS2 Odometry 消息的完整计算流程：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">参数</th>
<th style="text-align: left;">值</th>
</tr>
<tr>
<td style="text-align: left;">编码器 PPR</td>
<td style="text-align: left;">11</td>
</tr>
<tr>
<td style="text-align: left;">四倍频脉冲/圈</td>
<td style="text-align: left;">44</td>
</tr>
<tr>
<td style="text-align: left;">齿轮减速比 G</td>
<td style="text-align: left;">30</td>
</tr>
<tr>
<td style="text-align: left;">每圈轮子脉冲数</td>
<td style="text-align: left;">1320</td>
</tr>
<tr>
<td style="text-align: left;">采样周期 dt</td>
<td style="text-align: left;">20ms</td>
</tr>
<tr>
<td style="text-align: left;">车轮半径 r</td>
<td style="text-align: left;">0.06m</td>
</tr>
<tr>
<td style="text-align: left;">轮距 L</td>
<td style="text-align: left;">0.28m</td>
</tr>
</table>

In [ ]:
import numpy as np

# 里程计计算：从编码器脉冲到机器人位姿
PPR = 11
pulses_per_rev = PPR * 4  # 四倍频
gear_ratio = 30
pulses_per_wheel_rev = pulses_per_rev * gear_ratio  # 1320
dt = 0.02   # 采样周期 20ms
r = 0.06    # 车轮半径
L = 0.28    # 轮距

# 本次采样脉冲增量
delta_N_right = 36
delta_N_left = 34

# 步骤1：计算每个轮子的角速度
omega_r = (2 * np.pi * delta_N_right) / (pulses_per_wheel_rev * dt)
omega_l = (2 * np.pi * delta_N_left) / (pulses_per_wheel_rev * dt)
print(f'右轮角速度: {omega_r:.3f} rad/s')
print(f'左轮角速度: {omega_l:.3f} rad/s')

# 步骤2：计算线速度
v_r = r * omega_r
v_l = r * omega_l
print(f'右轮线速度: {v_r:.3f} m/s')
print(f'左轮线速度: {v_l:.3f} m/s')

# 步骤3：计算机器人本体速度
v_body = (v_r + v_l) / 2
omega_z = (v_r - v_l) / L
print(f'车身线速度: {v_body:.3f} m/s')
print(f'车身角速度: {omega_z:.3f} rad/s')

# 步骤4：计算位姿增量（假设当前朝向 theta=0）
theta = 0.0
delta_x = v_body * np.cos(theta) * dt
delta_y = v_body * np.sin(theta) * dt
delta_theta = omega_z * dt
print(f'位姿增量: dx={delta_x:.4f} m, dy={delta_y:.4f} m, dtheta={delta_theta:.4f} rad')

# 步骤5：构建四元数
q_z = np.sin(delta_theta / 2)
q_w = np.cos(delta_theta / 2)
print(f'四元数: z={q_z:.6f}, w={q_w:.6f}')

### 2.4 多传感器融合：EKF

纯轮式里程计存在**累积误差**（系统性：轮径标定误差、轮距偏差；非系统性：打滑、空转）。ROS2 生态提供 `robot_localization` 包，通过 **扩展卡尔曼滤波器（EKF）** 融合多种传感器：

<img src="../../images/ekf_sensor_fusion.png" alt="EKF传感器融合" style="display: block; margin-left: 0;" />

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">传感器</th>
<th style="text-align: left;">角色</th>
<th style="text-align: left;">优势</th>
<th style="text-align: left;">劣势</th>
</tr>
<tr>
<td style="text-align: left;">轮式里程计 <code>/odom</code></td>
<td style="text-align: left;">短跑专家</td>
<td style="text-align: left;">高频、平滑</td>
<td style="text-align: left;">长期累积漂移</td>
</tr>
<tr>
<td style="text-align: left;">IMU <code>/imu/data</code></td>
<td style="text-align: left;">姿态专家</td>
<td style="text-align: left;">高频、无漂移</td>
<td style="text-align: left;">位置二次积分漂移</td>
</tr>
<tr>
<td style="text-align: left;">视觉/激光里程计</td>
<td style="text-align: left;">校正专家</td>
<td style="text-align: left;">无累积漂移</td>
<td style="text-align: left;">特征缺失时退化</td>
</tr>
</table>

#### 动手实验：简易 EKF 融合演示

下面用一个简化的 1D EKF 演示如何融合轮式里程计（高频但有漂移）与 GPS（低频但无漂移）：

In [ ]:
import numpy as np

# 简化 1D EKF：融合轮式里程计(高频有漂移) + GPS(低频无漂移)
np.random.seed(42)

true_pos = 0.0
true_vel = 0.5  # 真实速度 0.5 m/s
dt = 0.1        # 里程计频率 10Hz
steps = 100

# EKF 状态：[位置, 速度]
x_est = np.array([0.0, 0.0])
P = np.diag([1.0, 1.0])
Q = np.diag([0.01, 0.01])
R_odom = 0.1
R_gps = 0.01

F = np.array([[1, dt], [0, 1]])
H_vel = np.array([[0, 1]])
H_pos = np.array([[1, 0]])

estimates = []

for k in range(steps):
    true_pos += true_vel * dt
    # 预测步
    x_pred = F @ x_est
    P_pred = F @ P @ F.T + Q
    # 更新步1: 里程计观测速度
    z_odom = true_vel + np.random.randn() * np.sqrt(R_odom) + 0.001*k
    y = z_odom - H_vel @ x_pred
    S = H_vel @ P_pred @ H_vel.T + R_odom
    K = P_pred @ H_vel.T / S
    x_est = x_pred + K.flatten() * y
    P = (np.eye(2) - K @ H_vel) @ P_pred
    # 更新步2: GPS观测位置 (每10步一次)
    if k % 10 == 0:
        z_gps = true_pos + np.random.randn() * np.sqrt(R_gps)
        y = z_gps - H_pos @ x_est
        S = H_pos @ P @ H_pos.T + R_gps
        K = P @ H_pos.T / S
        x_est = x_est + K.flatten() * y
        P = (np.eye(2) - K @ H_pos) @ P
    estimates.append(x_est.copy())

estimates = np.array(estimates)
print(f'真实最终位置: {true_pos:.2f} m')
print(f'EKF 估计位置: {estimates[-1, 0]:.2f} m')
print(f'EKF 估计速度: {estimates[-1, 1]:.3f} m/s (真实: {true_vel})')
print(f'位置估计误差: {abs(estimates[-1, 0] - true_pos):.4f} m')
print('EKF 成功融合了高频里程计(有漂移)和低频GPS(无漂移)，最终误差很小。')

### 2.5 在昇腾 NPU 上加速运动学计算

当需要对大量轨迹点进行运动学解算时，可以借助昇腾 NPU 的并行能力加速批量计算。

In [ ]:
import time

# 生成 100 万组 (v, omega) 轨迹点
N = 1_000_000
v_batch = np.random.uniform(0, 0.5, N)
omega_batch = np.random.uniform(-0.5, 0.5, N)
L = 0.28

# CPU 批量运动学解算
start = time.time()
v_left_cpu = v_batch - omega_batch * L / 2
v_right_cpu = v_batch + omega_batch * L / 2
cpu_time = time.time() - start
print(f'CPU 批量解算 {N} 组: {cpu_time*1000:.2f} ms')

# 尝试在 NPU 上加速
try:
    import torch
    import torch_npu
    if torch.npu.is_available():
        v_t = torch.from_numpy(v_batch).npu()
        omega_t = torch.from_numpy(omega_batch).npu()
        torch.npu.synchronize()
        start = time.time()
        v_left_npu = v_t - omega_t * L / 2
        v_right_npu = v_t + omega_t * L / 2
        torch.npu.synchronize()
        npu_time = time.time() - start
        print(f'NPU 批量解算 {N} 组: {npu_time*1000:.2f} ms')
        print(f'加速比: {cpu_time/npu_time:.1f}x')
        match = np.allclose(v_left_cpu, v_left_npu.cpu().numpy(), atol=1e-5)
        print(f'NPU 与 CPU 结果一致: {match}')
    else:
        print('NPU 不可用，跳过 NPU 加速演示')
except ImportError:
    print('torch_npu 未安装，跳过 NPU 加速演示')
    print('在昇腾云平台上运行此单元格可看到 NPU 加速效果')

---

## 3. 智能机器人SLAM开发

SLAM（Simultaneous Localization and Mapping，同步定位与地图构建）解决'我在哪里'和'环境长什么样'两个根本问题。

### 3.1 激光传感器原理

激光雷达基于 **飞行时间（ToF）** 测距：

$$\text{距离} = \frac{\text{光速} \times \text{时间差}}{2}$$

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">类型</th>
<th style="text-align: left;">激光束数</th>
<th style="text-align: left;">ROS2 话题</th>
<th style="text-align: left;">消息类型</th>
<th style="text-align: left;">适用场景</th>
</tr>
<tr>
<td style="text-align: left;">单线 LiDAR</td>
<td style="text-align: left;">1</td>
<td style="text-align: left;"><code>/scan</code></td>
<td style="text-align: left;"><code>sensor_msgs/LaserScan</code></td>
<td style="text-align: left;">室内移动机器人</td>
</tr>
<tr>
<td style="text-align: left;">多线 LiDAR</td>
<td style="text-align: left;">16/32/64/128</td>
<td style="text-align: left;"><code>/point_cloud</code></td>
<td style="text-align: left;"><code>sensor_msgs/PointCloud2</code></td>
<td style="text-align: left;">自动驾驶</td>
</tr>
</table>

### 3.2 常见激光 SLAM 方法

<img src="../../images/slam_methods_comparison.png" alt="SLAM方法对比" style="display: block; margin-left: 0;" />

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">方法</th>
<th style="text-align: left;">架构</th>
<th style="text-align: left;">优势</th>
<th style="text-align: left;">推荐场景</th>
</tr>
<tr>
<td style="text-align: left;"><strong>slam-toolbox</strong></td>
<td style="text-align: left;">图优化</td>
<td style="text-align: left;">终身地图、RViz 插件、生产级</td>
<td style="text-align: left;">长期运行、大规模建图</td>
</tr>
<tr>
<td style="text-align: left;"><strong>Cartographer</strong></td>
<td style="text-align: left;">双层优化</td>
<td style="text-align: left;">高精度、低漂移</td>
<td style="text-align: left;">大型复杂室内</td>
</tr>
<tr>
<td style="text-align: left;"><strong>Gmapping</strong></td>
<td style="text-align: left;">粒子滤波</td>
<td style="text-align: left;">简单易懂</td>
<td style="text-align: left;">小型室内、教学</td>
</tr>
</table>

### 3.3 导航方法：全局规划器与局部规划器

<img src="../../images/nav2_architecture.png" alt="Nav2架构" style="display: block; margin-left: 0;" />

**全局规划器**：基于全局地图生成参考路径
- Dijkstra：保证最优但遍历所有节点
- A*：引入启发式函数，搜索效率高
- SmacPlanner：支持运动学约束

**局部规划器**：实时避障与轨迹跟踪
- DWA：动态窗口法，速度空间采样
- TEB：时间弹性带，平滑且满足运动学约束
- RegulatedPurePursuit：增强型纯追踪

#### 动手实验：A* 路径规划

下面实现一个完整的 A* 算法，在栅格地图上规划路径：

In [ ]:
import numpy as np
import heapq
import matplotlib.pyplot as plt

class AStarPlanner:
    def __init__(self, grid):
        self.grid = grid
        self.rows, self.cols = grid.shape
    def heuristic(self, a, b):
        return np.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2)
    def get_neighbors(self, node):
        neighbors = []
        for dx, dy in [(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(-1,1),(1,-1),(1,1)]:
            nx, ny = node[0]+dx, node[1]+dy
            if 0 <= nx < self.rows and 0 <= ny < self.cols and self.grid[nx, ny] == 0:
                neighbors.append((nx, ny))
        return neighbors
    def plan(self, start, goal):
        open_set = [(0, start)]
        came_from = {}
        g_score = {start: 0}
        while open_set:
            _, current = heapq.heappop(open_set)
            if current == goal:
                path = [current]
                while current in came_from:
                    current = came_from[current]
                    path.append(current)
                return path[::-1]
            for neighbor in self.get_neighbors(current):
                dx = neighbor[0] - current[0]
                dy = neighbor[1] - current[1]
                tentative_g = g_score[current] + np.sqrt(dx**2 + dy**2)
                if neighbor not in g_score or tentative_g < g_score[neighbor]:
                    came_from[neighbor] = current
                    g_score[neighbor] = tentative_g
                    f = tentative_g + self.heuristic(neighbor, goal)
                    heapq.heappush(open_set, (f, neighbor))
        return []

grid = np.zeros((15, 15))
grid[0, :] = 1; grid[-1, :] = 1; grid[:, 0] = 1; grid[:, -1] = 1
grid[3:6, 3:10] = 1
grid[8:12, 5:8] = 1
grid[5:10, 10:12] = 1

planner = AStarPlanner(grid)
start, goal = (1, 1), (13, 13)
path = planner.plan(start, goal)
print(f'A* 规划路径长度: {len(path)} 步')
print(f'起点: {start}, 终点: {goal}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(grid, cmap='binary', origin='lower')
if path:
    py, px = zip(*path)
    ax.plot(px, py, 'r-', linewidth=2, marker='o', markersize=3, label='A* Path')
ax.plot(start[1], start[0], 'go', markersize=12, label='Start')
ax.plot(goal[1], goal[0], 'b^', markersize=12, label='Goal')
ax.set_title('A* Path Planning Result', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

### 3.4 AMCL 自适应蒙特卡洛定位

<img src="../../images/amcl_particle_filter.png" alt="AMCL粒子滤波" style="display: block; margin-left: 0;" />

AMCL 在地图中撒出代表'可能位姿'的粒子，用激光观测为粒子打分、重采样，粒子云逐渐收拢到真实位姿。

> **关键操作**：导航前必须用 `2D Pose Estimate` 给定初始位姿，否则粒子无从收敛。

#### 动手实验：粒子滤波定位仿真

下面用一个简化的粒子滤波演示在 2D 环境中的定位过程：

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
true_pos = np.array([5.0, 5.0])
n_particles = 500
particles = np.random.uniform([0, 0], [10, 10], (n_particles, 2))
weights = np.ones(n_particles) / n_particles

beacons = np.array([[0, 0], [10, 0], [5, 10]])
def observe(pos):
    return np.sqrt(np.sum((beacons - pos)**2, axis=1))

true_obs = observe(true_pos)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
steps_to_show = [0, 5, 20]

for step in range(25):
    for i, p in enumerate(particles):
        obs = observe(p)
        weights[i] = np.exp(-np.sum((obs - true_obs)**2) / 0.5)
    weights /= weights.sum()
    if step in steps_to_show:
        ax = axes[steps_to_show.index(step)]
        ax.scatter(particles[:, 0], particles[:, 1], c='red', s=5, alpha=0.3, label='Particles')
        ax.plot(true_pos[0], true_pos[1], 'b^', markersize=15, label='True Pose')
        est = np.average(particles, weights=weights, axis=0)
        ax.plot(est[0], est[1], 'g*', markersize=15, label='Estimated Pose')
        ax.set_title(f'Step {step}', fontsize=13)
        ax.legend(fontsize=9)
        ax.set_xlim(-1, 11); ax.set_ylim(-1, 11)
    indices = np.random.choice(n_particles, n_particles, p=weights)
    particles = particles[indices] + np.random.randn(n_particles, 2) * 0.3
    weights = np.ones(n_particles) / n_particles

est_pos = np.average(particles, weights=weights, axis=0)
print(f'真实位置: ({true_pos[0]}, {true_pos[1]})')
print(f'估计位置: ({est_pos[0]:.2f}, {est_pos[1]:.2f})')
print(f'定位误差: {np.linalg.norm(est_pos - true_pos):.3f} m')
plt.suptitle('Particle Filter Localization Convergence', fontsize=14)
plt.tight_layout()
plt.show()

---

## 小结

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">概念</th>
<th style="text-align: left;">一句话理解</th>
</tr>
<tr>
<td style="text-align: left;">三大核心模块</td>
<td style="text-align: left;">硬件=身体，软件=神经系统，算法=大脑</td>
</tr>
<tr>
<td style="text-align: left;">差动驱动运动学</td>
<td style="text-align: left;">v_left = v - wL/2, v_right = v + wL/2</td>
</tr>
<tr>
<td style="text-align: left;">/cmd_vel</td>
<td style="text-align: left;">自上而下的运动指令通道（Twist 消息）</td>
</tr>
<tr>
<td style="text-align: left;">/odom</td>
<td style="text-align: left;">自下而上的状态反馈通道（Odometry 消息）</td>
</tr>
<tr>
<td style="text-align: left;">EKF 融合</td>
<td style="text-align: left;">预测-更新循环，融合多传感器取长补短</td>
</tr>
<tr>
<td style="text-align: left;">SLAM</td>
<td style="text-align: left;">同步定位与地图构建，slam-toolbox 推荐</td>
</tr>
<tr>
<td style="text-align: left;">AMCL</td>
<td style="text-align: left;">粒子滤波定位，需给定初始位姿</td>
</tr>
<tr>
<td style="text-align: left;">Nav2</td>
<td style="text-align: left;">全局规划 + 局部控制 + 恢复行为</td>
</tr>
<tr>
<td style="text-align: left;">昇腾 NPU</td>
<td style="text-align: left;">为感知类 AI 任务提供算力，与导航 CPU 异构分工</td>
</tr>
</table>

---

## 课后练习

请根据本节课程学习内容完成以下题目进行自测。

**第1题**（单选题）智能机器人系统的三大核心模块是什么？


- A. 硬件系统、软件系统、智能算法
- B. 传感器、电机、控制器
- C. CPU、GPU、内存
- D. 机械臂、底盘、摄像头


In [ ]:
q1 = ''  # 填入你的选项，如 'B'，修改后务必运行本单元格（Shift+Enter）
print(f'第1题答案已记录：{q1}' if q1 else '请填入答案并运行本单元格')

**第2题**（单选题）ROS2 相比 ROS1 的核心改进是采用了什么作为底层通信中间件？


- A. TCP/IP
- B. DDS（数据分发服务）
- C. HTTP
- D. WebSocket


In [ ]:
q2 = ''  # 填入你的选项，如 'B'，修改后务必运行本单元格（Shift+Enter）
print(f'第2题答案已记录：{q2}' if q2 else '请填入答案并运行本单元格')

**第3题**（单选题）差动驱动机器人运动学公式中，右轮线速度的计算公式是？


- A. v_right = v - wL/2
- B. v_right = v + wL/2
- C. v_right = v * wL
- D. v_right = v / wL


In [ ]:
q3 = ''  # 填入你的选项，如 'B'，修改后务必运行本单元格（Shift+Enter）
print(f'第3题答案已记录：{q3}' if q3 else '请填入答案并运行本单元格')

**第4题**（单选题）ROS2 中 /cmd_vel 话题发布的消息类型是？


- A. nav_msgs/Odometry
- B. geometry_msgs/Twist
- C. sensor_msgs/LaserScan
- D. std_msgs/Float32


In [ ]:
q4 = ''  # 填入你的选项，如 'B'，修改后务必运行本单元格（Shift+Enter）
print(f'第4题答案已记录：{q4}' if q4 else '请填入答案并运行本单元格')

**第5题**（单选题）ROS2 中 /odom 话题发布的消息类型是？


- A. geometry_msgs/Twist
- B. nav_msgs/Odometry
- C. sensor_msgs/Imu
- D. tf2_msgs/TFMessage


In [ ]:
q5 = ''  # 填入你的选项，如 'B'，修改后务必运行本单元格（Shift+Enter）
print(f'第5题答案已记录：{q5}' if q5 else '请填入答案并运行本单元格')

**第6题**（单选题）纯轮式里程计的根本缺陷是什么？


- A. 计算速度太慢
- B. 累积误差随时间放大
- C. 需要太多内存
- D. 只能用于室内


In [ ]:
q6 = ''  # 填入你的选项，如 'B'，修改后务必运行本单元格（Shift+Enter）
print(f'第6题答案已记录：{q6}' if q6 else '请填入答案并运行本单元格')

**第7题**（单选题）EKF 的预测步基于什么？


- A. 传感器观测
- B. 上一时刻最优状态估计和运动学模型
- C. GPS数据
- D. 激光雷达数据


In [ ]:
q7 = ''  # 填入你的选项，如 'B'，修改后务必运行本单元格（Shift+Enter）
print(f'第7题答案已记录：{q7}' if q7 else '请填入答案并运行本单元格')

**第8题**（单选题）slam-toolbox 采用的架构是？


- A. 粒子滤波
- B. 图优化
- C. 卡尔曼滤波
- D. 深度学习


In [ ]:
q8 = ''  # 填入你的选项，如 'B'，修改后务必运行本单元格（Shift+Enter）
print(f'第8题答案已记录：{q8}' if q8 else '请填入答案并运行本单元格')

**第9题**（单选题）AMCL 定位前必须先做什么操作？


- A. 启动激光雷达
- B. 用 2D Pose Estimate 给定初始位姿
- C. 保存地图
- D. 设置目标点


In [ ]:
q9 = ''  # 填入你的选项，如 'B'，修改后务必运行本单元格（Shift+Enter）
print(f'第9题答案已记录：{q9}' if q9 else '请填入答案并运行本单元格')

**第10题**（单选题）Nav2 导航栈中，全局规划器默认使用的算法是？


- A. A*
- B. Dijkstra (NavfnPlanner)
- C. RRT
- D. DWA


In [ ]:
q10 = ''  # 填入你的选项，如 'B'，修改后务必运行本单元格（Shift+Enter）
print(f'第10题答案已记录：{q10}' if q10 else '请填入答案并运行本单元格')

**第11题**（单选题）单线激光雷达在 ROS2 中发布的话题和消息类型是？


- A. /point_cloud, PointCloud2
- B. /scan, LaserScan
- C. /image, Image
- D. /odom, Odometry


In [ ]:
q11 = ''  # 填入你的选项，如 'B'，修改后务必运行本单元格（Shift+Enter）
print(f'第11题答案已记录：{q11}' if q11 else '请填入答案并运行本单元格')

**第12题**（单选题）昇腾 NPU 在智能机器人系统中的主要角色是？


- A. 运行 SLAM 算法
- B. 为感知类 AI 任务提供算力
- C. 控制电机
- D. 发布 /odom 话题


In [ ]:
q12 = ''  # 填入你的选项，如 'B'，修改后务必运行本单元格（Shift+Enter）
print(f'第12题答案已记录：{q12}' if q12 else '请填入答案并运行本单元格')

**全部作答完成后，运行下方代码查看批改结果：**


In [ ]:
import sys
from pathlib import Path

for candidate in (
    Path.cwd() / 'answer',
    Path.cwd() / '08_robot_dev' / 'answer',
):
    if candidate.exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise FileNotFoundError('Cannot find answer directory')
from grade_01 import grade
grade(globals())

---

## 参考资料

- [ROS2 官方文档](https://docs.ros.org/en/rolling/)
- [SLAM Toolbox](https://github.com/SteveMacenski/slam_toolbox)
- [Nav2 导航框架](https://navigation.ros.org/)
- [robot_localization (EKF)](https://github.com/cra-ros/robot_localization)
- [昇腾社区 - CANN 文档](https://hiascend.com/document)

> 下一节：[02_drl_obstacle_avoidance.ipynb](./02_drl_obstacle_avoidance.ipynb)